In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-10 02:25:25--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip


Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip    0%[                    ]       0  --.-KB/s               

book-crossings.zip   14%[=>                  ]   3.56M  17.8MB/s               

book-crossings.zip   29%[====>               ]   7.23M  18.0MB/s               

book-crossings.zip   45%[========>           ]  11.23M  18.7MB/s               

... (3 outputs left out)

book-crossings.zip  100%[===================>]  24.88M  19.9MB/s    in 1.3s    

2026-11-10 02:25:27 (19.9 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]



Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     


  inflating: BX-Books.csv            


  inflating: BX-Users.csv            

In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# Keep only users with at least 200 ratings and books with at least 100 ratings
# so that the neighbours are statistically meaningful.
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

df = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]
df = df[df['isbn'].isin(book_counts[book_counts >= 100].index)]

# Attach titles; the same title may exist under several ISBNs.
df = df.merge(df_books, on='isbn')
df = df.drop_duplicates(['title', 'user'])

# Rows: books, columns: users, values: ratings (0 when not rated).
book_user = df.pivot(index='title', columns='user', values='rating').fillna(0)

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(csr_matrix(book_user.values))
book_user.shape

(673, 888)

In [5]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  distances, indices = model.kneighbors(book_user.loc[[book]].values, n_neighbors=6)
  # Skip the first neighbour (the book itself); list from least to most similar.
  similar = [[book_user.index[i], float(d)] for i, d in zip(indices[0][1:], distances[0][1:])]
  recommended_books = [book, similar[::-1]]

  return recommended_books

In [6]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]
You passed the challenge! 🎉🎉🎉🎉🎉
